In [1]:
import warnings
import pandas as pd
import numpy as np

warnings.filterwarnings("ignore", category=UserWarning, module="openpyxl")

transaction_path = r"G:\My Drive\New_Working_Files\KNIME\CTU_2025\Payment_Adonc\Grocery_Refund\Transaction\Grocery_Transaction_refund.csv"
df = pd.read_csv(transaction_path, low_memory=False)

In [2]:
df=df[df['marketplace_id'] !="marketplace_id"]
df = df[df['refund_init_date'] !="refund_init_date"]
df['refund_init_date'] =df['refund_init_date'].astype("datetime64[ns]")
df['p_refund_count'] = pd.to_numeric(df["p_refund_count"], errors ='coerce')
df=df.sort_values(by=["refund_init_date"])
df[df['refund_init_date'].isna()]

,marketplace_id,refund_year,refund_month,refund_week,refund_init_date,refund_complete_date,refund_status,refund_final_status_updated,mr_refund_flag,refund_mode,...,refund_reason,refund_reason_flag,payment_type_final,cx_count,rf_orders,p_refund_count,c_refund_count,orders,total_refund_amount,is_shopsy_order


In [ ]:
print(df.dtypes)

In [ ]:
df = df[df["Is_shopsy_order"] != True]

##df[df['marketplace_id']!="marketplace_id"]
df['refund_init_date'] = df['refund_init_date'].astype(str).str.replace(r'[\x00-\x1F\u200B-\u200F\uFEFF\u00A0]', '', regex=True).str.strip()

df["refund_init_date"] = pd.to_datetime(df["refund_init_date"], format="%d-%m-%Y", errors="coerce")


df["p_refund_count"] = pd.to_numeric(df["p_refund_count"], errors = 'coerce')
df["refund_init_date"] = pd.to_datetime(df["refund_init_date"], errors='coerce', format='%d-%m-%Y')

within_24 = ["2. 0 To 4 Hrs", "3. 4 To 8 Hrs", "4. 8 To 24 Hrs"]
df["refund_init_completion_bucket_flag"]=df["refund_init_completion_bucket"].apply(
    lambda x: "within_24" if x in within_24 else "greater_24"
)

filtered_data = df[df["marketplace_id"] == "GROCERY"].copy()

transaction_order = ['Completed', 'Failed', 'Cancelled', 'Stuck', 'Clone']
filtered_data["refund_final_status_updated"] = pd.Categorical(filtered_data["refund_final_status_updated"], categories=transaction_order, ordered=True)

filtered_data["refund_init_date"] = pd.to_datetime(filtered_data["refund_init_date"], errors='coerce', format='%d-%m-%Y')

refund_final_status = (
    filtered_data.groupby(["refund_final_status_updated", "refund_init_completion_bucket_flag", "refund_init_date"], observed=False)["p_refund_count"]
    .sum()
    .reset_index()
    .rename(columns={"p_refund_count": "refund_status_count"})
)
pivot_source = refund_final_status.pivot_table(index=["refund_final_status_updated","refund_init_completion_bucket_flag"],
                                                columns = "refund_init_date",
                                                values = "refund_status_count",
                                                aggfunc='sum',
                                                observed= False
                                              ).fillna(0)
                                              
pivot_source.columns=pd.to_datetime(pivot_source.columns, errors='coerce')
pivot_source = pivot_source.sort_index(axis=1)
pivot_source.columns = pivot_source.columns.strftime('%d-%m-%Y')
pivot_source.reset_index(inplace=True)

#print(pivot_source)

####-------------------------###

df["p_refund_count"] = pd.to_numeric(df["p_refund_count"], errors = 'coerce')
df["refund_init_date"] = pd.to_datetime(df["refund_init_date"], errors='coerce', format='%d-%m-%Y')

filtered_data1 = df[df["marketplace_id"] == "GROCERY"].copy()
transaction_order5 = ['Completed', 'Failed', 'Cancelled', 'Stuck', 'Clone']
filtered_data1["refund_final_status_updated"] = pd.Categorical(filtered_data1["refund_final_status_updated"], categories=transaction_order, ordered=True)

filtered_data1["refund_init_date"] = pd.to_datetime(df["refund_init_date"], errors='coerce', format='%d-%m-%Y')

refund_final_status1 = (
    filtered_data1.groupby(["refund_final_status_updated",  "refund_init_date"], observed=False)["p_refund_count"]
    .sum()
    .reset_index()
    .rename(columns={"p_refund_count": "refund_status_count1"})
)
pivot_source1 = refund_final_status1.pivot_table(index=["refund_final_status_updated"],
                                                columns = "refund_init_date",
                                                values = "refund_status_count1",
                                                aggfunc='sum',
                                                observed= False
                                              ).fillna(0)
                                              
pivot_source1.columns=pd.to_datetime(pivot_source1.columns, errors='coerce')
pivot_source1 = pivot_source1.sort_index(axis=1)
pivot_source1.columns = pivot_source1.columns.strftime('%d-%m-%Y')
pivot_source1.reset_index(inplace=True)

#print(pivot_source1)


##------------------##
##refund_reason_flag
df["refund_reason_group"] = df["refund_reason_flag"].apply(
    lambda x: "Cancel_RTO" if x in ["Cancellation", "RTO"] 
    else x if x in ["Return","Adonc"]
    else "Others"
)

df["p_refund_count"] = pd.to_numeric(df["p_refund_count"], errors = 'coerce')
df["refund_init_date"] = pd.to_datetime(df["refund_init_date"], errors='coerce', format='%d-%m-%Y')

filtered_data2 = df[df["marketplace_id"] == "GROCERY"].copy()
transaction_order1 = ['Cancel_RTO', 'Return', 'Adonc', 'Others']
filtered_data2["refund_reason_group"] = pd.Categorical(filtered_data2["refund_reason_group"], categories=transaction_order1, ordered=True)

refund_reason_flag1 = (filtered_data2.groupby(["refund_reason_group", "refund_init_date"], observed=False)["p_refund_count"].sum().reset_index()
                .rename(columns={"p_refund_count": "refund_reason_group_count"}))

pivot_source_reason = refund_reason_flag1.pivot_table(index="refund_reason_group",
                                                columns = "refund_init_date",
                                                values = "refund_reason_group_count",
                                                aggfunc='sum',
                                                observed=False     
                                                 ).fillna(0)
                                              
pivot_source_reason.columns=pd.to_datetime(pivot_source_reason.columns, errors='coerce')
pivot_source_reason = pivot_source_reason.sort_index(axis=1)
pivot_source_reason.columns = pivot_source_reason.columns.strftime('%d-%m-%Y')
pivot_source_reason.reset_index(inplace=True)

#print(pivot_source_reason)

##------------------##
##refund_reason_flag
df["payment_type_final_flag"] = df["payment_type_final"].apply(
    lambda x: "Pre-paid" if x in ["Pre-paid", "EGV/Wallet/Coin"] 
    else x 
)

filtered_data3 = df[df["marketplace_id"] == "GROCERY"].copy()
transaction_order2 = ['Pre-paid', 'COD', 'Multi-payment', 'NA']
filtered_data3["payment_type_final_flag"] = pd.Categorical(filtered_data3["payment_type_final_flag"], categories=transaction_order2, ordered=True)

refund_summary_by_payment_type = (filtered_data3.groupby(["payment_type_final_flag", "refund_init_date"], observed=False)["p_refund_count"].sum().reset_index()
                .rename(columns={"p_refund_count": "payment_type_final_count"}))

pivot_source_payment = refund_summary_by_payment_type.pivot_table(index="payment_type_final_flag",
                                                columns = "refund_init_date",
                                                values = "payment_type_final_count",
                                                aggfunc='sum',
                                                observed=False     
                                                 ).fillna(0)
                                              
pivot_source_payment.columns=pd.to_datetime(pivot_source_payment.columns, errors='coerce')
pivot_source_payment = pivot_source_payment.sort_index(axis=1)
pivot_source_payment.columns = pivot_source_payment.columns.strftime('%d-%m-%Y')
pivot_source_payment.reset_index(inplace=True)

##print(pivot_source_payment)

##------------------##
##ARN_issue_flag

filtered_data4 = df[df["marketplace_id"] == "GROCERY"].copy()
transaction_order3 = ['ARN_GENERATED', 'ARN_NOT_GENERATED', 'ARN_NOT_APPLICABLE']
filtered_data4["ARN_ISSUE_FLAG"] = pd.Categorical(filtered_data4["ARN_ISSUE_FLAG"], categories=transaction_order3, ordered=True)

ARN_issue_flag_summary = (filtered_data4.groupby(["ARN_ISSUE_FLAG", "refund_init_date"], observed=False)["p_refund_count"].sum().reset_index()
                .rename(columns={"p_refund_count": "ARN_ISSUE_count"}))

pivot_source_ARN = ARN_issue_flag_summary.pivot_table(index="ARN_ISSUE_FLAG",
                                                columns = "refund_init_date",
                                                values = "ARN_ISSUE_count",
                                                aggfunc='sum',
                                                observed=False     
                                                 ).fillna(0)
                                              
pivot_source_ARN.columns=pd.to_datetime(pivot_source_ARN.columns, errors='coerce')
pivot_source_ARN = pivot_source_ARN.sort_index(axis=1)
pivot_source_ARN.columns = pivot_source_ARN.columns.strftime('%d-%m-%Y')
pivot_source_ARN.reset_index(inplace=True)

##print(pivot_source_ARN)

##------------------##
##Refund_overall

filtered_data5 = df[df["marketplace_id"] == "GROCERY"].copy()
Refund_overall_summary = (filtered_data5.groupby(["refund_init_date"], observed=False)["p_refund_count"].sum().reset_index()
                .rename(columns={"p_refund_count": "Overall_refund_count"}))

Refund_overall_summary = Refund_overall_summary.sort_values("refund_init_date")
pivot_Refund_overall_summary=Refund_overall_summary.set_index("refund_init_date").T
                                              
pivot_Refund_overall_summary.columns=pd.to_datetime(pivot_Refund_overall_summary.columns, errors='coerce')
pivot_Refund_overall_summary = pivot_Refund_overall_summary.sort_index(axis=1)
pivot_Refund_overall_summary.columns = pivot_Refund_overall_summary.columns.strftime('%d-%m-%Y')
pivot_Refund_overall_summary.reset_index(inplace=True)

#print(pivot_Refund_overall_summary)



##---Void_Transactions-----

##Refund_overall

filtered_data6 = df[df["marketplace_id"] == "GROCERY"].copy()
Refund_void_summary = (filtered_data6.groupby(["void_flag", "refund_init_date"], observed=False)["p_refund_count"].sum().reset_index()
                .rename(columns={"p_refund_count": "refund_void_count"}))

pivot_void_summary = Refund_void_summary.pivot_table(index="void_flag",
                                                columns = "refund_init_date",
                                                values = "refund_void_count",
                                                aggfunc='sum',
                                                observed=False     
                                                 ).fillna(0)
                                              
pivot_void_summary.columns=pd.to_datetime(pivot_void_summary.columns, errors='coerce')
pivot_void_summary = pivot_void_summary.sort_index(axis=1)
pivot_void_summary.columns = pivot_void_summary.columns.strftime('%d-%m-%Y')
pivot_void_summary.reset_index(inplace=True)

##print(pivot_void_summary)

#####------------------------
##Refund_modes

filtered_data7 = df[df["marketplace_id"] == "GROCERY"].copy()
transaction_order4 = ["UPI","UPI_COLLECT","UPI_INTENT","FK_UPI","PHONEPE","NETBANKING","NEFT","IMPS","RTGS","CREDIT_CARD","CREDIT_CARD_EMI","DEBIT_CARD","DYNAMIC_QR","COIN",
        "E_GIFT_VOUCHER","QC_EGV","EGV","EGV_WALLET","WALLET","FLIPKART_FINANCE","InvalidRefundMode","LOAN","SUPERPAY_LATER"]

filtered_data7["refund_mode"] = filtered_data7["refund_mode"].apply(
    lambda x: x if x in transaction_order4 else "Others"
)
category_list = transaction_order4 + ["Others"]
filtered_data7["refund_mode"] = pd.Categorical(filtered_data7["refund_mode"], categories=category_list, ordered=True)


refund_mode_flag_summary = (filtered_data7.groupby(["refund_mode", "refund_init_date"], observed=False)["p_refund_count"].sum().reset_index()
                .rename(columns={"p_refund_count": "refund_mode_count"}))

pivot_source_refund_mode = refund_mode_flag_summary.pivot_table(index="refund_mode",
                                                columns = "refund_init_date",
                                                values = "refund_mode_count",
                                                aggfunc='sum',
                                                observed=False     
                                                 ).fillna(0)
                                              
pivot_source_refund_mode.columns=pd.to_datetime(pivot_source_refund_mode.columns, errors='coerce')
pivot_source_refund_mode = pivot_source_refund_mode.sort_index(axis=1)
pivot_source_refund_mode.columns = pivot_source_refund_mode.columns.strftime('%d-%m-%Y')
pivot_source_refund_mode.reset_index(inplace=True)

output_path = r"G:\My Drive\New_Working_Files\KNIME\CTU_2025\Payment_Adonc\Grocery_refund_transaction.xlsx"
with pd.ExcelWriter(output_path, engine = 'openpyxl') as writer:
    pivot_source.to_excel(writer, sheet_name ='refund_final_status_updated', index=False)
    pivot_source1.to_excel(writer, sheet_name ='refund_final_status', index=False)
    pivot_source_reason.to_excel(writer, sheet_name ='refund_reason', index=False )
    pivot_source_payment.to_excel(writer, sheet_name ='refund_source_payment', index=False )
    pivot_source_ARN.to_excel(writer, sheet_name = 'refund_ARN_flag', index=False)
    pivot_void_summary.to_excel(writer, sheet_name = 'pivot_Refund_Void', index=False)
    pivot_source_refund_mode.to_excel(writer, sheet_name = 'refund_mode_ssi', index=False)
    pivot_Refund_overall_summary.to_excel(writer, sheet_name ='Overall_count, index =False')



print("Excel file successfully saved:",output_path)

Excel file successfully saved: G:\My Drive\New_Working_Files\KNIME\CTU_2025\Payment_Adonc\Grocery_refund_transaction.xlsx


In [2]:
raw_df = pd.read_csv(transaction_path, low_memory=False)
print(raw_df.columns.tolist())
print(raw_df["refund_init_date"].head(10))

['refund_year', 'refund_month', 'refund_week', 'refund_init_date', 'refund_complete_date', 'refund_status', 'refund_final_status_updated', 'mr_refund_flag', 'refund_mode', 'auth_state', 'payment_mode', 'instrument_type', 'payment_instrument', 'ref_amount_flag', 'transaction_source', 'pg_id', 'promise_sla_refund_bucket', 'refund_init_completion_bucket', 'overall_ageing_Stuck', 'refund_complition_bucket', 'ARN_ISSUE_FLAG', 'void_flag', 'refund_reason', 'refund_reason_flag', 'payment_type_final', 'cx_count', 'rf_orders', 'p_refund_count', 'c_refund_count', 'total_refund_amount']
0    01-03-2025
1    01-03-2025
2    01-03-2025
3    01-03-2025
4    01-03-2025
5    01-03-2025
6    01-03-2025
7    01-03-2025
8    01-03-2025
9    01-03-2025
Name: refund_init_date, dtype: object
